# Дообучение YOLO: больше болезней и культур (в том числе пшеница)
Запускайте в Google Colab: **Среда выполнения → Сменить тип → GPU (T4)**, затем ячейки сверху вниз.

Идея: взять ваш прежний датасет (яблоня, томат, кукуруза и др.) и добавить датасеты с пшеницей и другими культурами, склеить их и обучить заново.
Проверьте лицензии всех датасетов на их страницах перед использованием в проекте.

In [ ]:
!pip -q install -U ultralytics roboflow pyyaml

In [ ]:
from google.colab import files
files.upload()   # выберите training/merge_datasets.py из проекта

## Данные
1. **Ваш прежний датасет**, на котором обучалась модель с классами Apple, Tomato, Corn…: загрузите архив в Colab или на Google Drive и распакуйте в папку `ds_old`.
2. **Пшеница и рис** (Roboflow Universe, 14 классов: бурая, жёлтая и чёрная ржавчина, септориоз, мучнистая роса, блэст, фузариоз и др.).
   Нужен бесплатный ключ: roboflow.com → Settings → API Keys. Номер версии смотрите на странице датасета; если код ниже не сработает, на странице нажмите *Download Dataset → YOLOv8* и вставьте готовый код оттуда.
3. Любые другие датасеты в формате YOLO добавляйте так же отдельными папками.

In [ ]:
# 1) прежний датасет (пример: архив на Google Drive)
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/мой_датасет.zip -d ds_old

# 2) пшеница и рис
from roboflow import Roboflow
rf = Roboflow(api_key="ВАШ_КЛЮЧ")
rf.workspace("yolodataset-hpzyu").project("dataset-8yhtx").version(1).download("yolov8", location="ds_wheat")

## Склейка
В `drop` перечислите классы, которые не нужны (например, рис), в `rename` можно привести одинаковые классы к одному названию.
Внизу таблицы будет видно, каких классов мало: меньше 100 примеров для обучения значит, что модель распознает их плохо.

In [ ]:
import json
json.dump({"drop": [], "rename": {}}, open("map.json", "w"), ensure_ascii=False)
!python merge_datasets.py --out merged --map map.json ds_old ds_wheat

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")        # yolo11n быстрее, yolo11s точнее
model.train(data="merged/data.yaml", epochs=100, imgsz=640, batch=16, patience=25, project="runs", name="agro", plots=True)

## Проверка по классам
Смотрите на столбец по каждому классу: общий mAP может быть приличным, а отдельные болезни не распознаваться.

In [ ]:
best = YOLO("runs/agro/weights/best.pt")
m = best.val(data="merged/data.yaml", split="val")
print("mAP50:", round(m.box.map50, 3), " mAP50-95:", round(m.box.map, 3))
for i, v in enumerate(m.box.maps):
    print(f"{best.names[i]:40} {v:.3f}")

## Забрать модель
Скачайте `best.pt`, положите в папку проекта `models/` под именем `best.pt` (вместо старого) и выполните `docker compose up --build -d`.
Список классов сервер прочитает сам: он виден по адресу `/api/v1/models`.

In [ ]:
from google.colab import files
files.download("runs/agro/weights/best.pt")